In [1]:
print('nurc-tts')

nurc-tts


In [2]:
!pip install -q datasets
!pip install -q num2words
!pip install -q librosa
!pip install -q soundfile
!pip install -q torchcodec
!pip install torch

  Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl (532.3 MB)
Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl (366.2 MB)


In [3]:
from datasets import load_dataset
import pandas as pd

In [4]:
df_sp = load_dataset('nilc-nlp/nurc_tts', split="sao_paulo")
df_re = load_dataset('nilc-nlp/nurc_tts', split="recife")

Resolving data files:   0%|          | 0/125 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/181 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/125 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/181 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/139 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/125 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/181 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/125 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/181 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
import os
import librosa
import soundfile as sf
from datasets import Audio

# 1. Configuration
TARGET_SR = 24000
OUTPUT_DIR = "/mnt/c/tmp/nurc-tts-resample-24khz"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def resample_and_store(batch, indices, subset):
    new_paths = []

    os.makedirs(OUTPUT_DIR+f"/{subset}", exist_ok=True)

    for i, audio_data in enumerate(batch["audio"]):

        # Define local filename based on index
        file_path = os.path.join(OUTPUT_DIR, subset, f"audio_{indices[i]}.wav")

        if os.path.exists(file_path):
          if os.path.getsize(file_path) > 44:
              new_paths.append(file_path)
              continue

        try:
            # dataset['audio'] returns a dict with 'array' and 'sampling_rate'
            original_array = audio_data["array"]
            original_sr = audio_data["sampling_rate"]

            # Perform the high-quality resampling
            resampled_array = librosa.resample(
                original_array,
                orig_sr=original_sr,
                target_sr=TARGET_SR
            )

            # Save to disk (soundfile handles the numpy array directly)
            sf.write(file_path, resampled_array, TARGET_SR)

            new_paths.append(file_path)

        except Exception as e:
            print(f"Skipping index {indices[i]} due to error: {e}")
            new_paths.append(None)

    # Return the list of paths to overwrite the 'audio' column
    return {"audio": new_paths}

# 2. Process the dataset
# librosa is CPU intensive, so adjust num_proc based on your machine
df_sp = df_sp.map(
    resample_and_store,
    batched=True,
    with_indices=True,
    num_proc=16,
    fn_kwargs={"subset": "SP"}
)

# 3. Cast the column so the Hub recognizes these as audio files
df_sp = df_sp.cast_column("audio", Audio(sampling_rate=TARGET_SR))

# Verify
print(f"Final SR: {df_sp[0]['audio']['sampling_rate']} Hz")


df_re = df_re.map(
    resample_and_store,
    batched=True,
    with_indices=True,
    num_proc=16,
    fn_kwargs={"subset": "RE"}
)

# 3. Cast the column so the Hub recognizes these as audio files
df_re = df_re.cast_column("audio", Audio(sampling_rate=TARGET_SR))

# Verify
print(f"Final SR: {df_re[0]['audio']['sampling_rate']} Hz")


In [ ]:
from huggingface_hub import notebook_login

# Execute esta célula para fazer login no Hugging Face
notebook_login()

In [ ]:
import re
from num2words import num2words

def replace_numbers_with_words(text):
    # Regular expression to find numbers in the text
    ret = re.sub(r'\d+', lambda x: num2words(int(x.group(0)), lang='pt_BR'), text)
    return ret.replace('%', ' porcento')

# Apply the transformation to the 'text' column
df_sp = df_sp.map(lambda x: {'text': replace_numbers_with_words(x['text'])})
df_re = df_re.map(lambda x: {'text': replace_numbers_with_words(x['text'])})

In [9]:
from datasets import DatasetDict
dataset_dict = DatasetDict({"sao_paulo": df_sp, "recife": df_re})
print(dataset_dict)


DatasetDict({
    sao_paulo: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 120621
    })
    recife: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 200295
    })
})


In [ ]:
dataset_dict.push_to_hub("sidleal/nurc_tts_24khz", private=False)

#old ------------------------------

In [ ]:
splits = ['sao_paulo', 'recife']
all_metadata = []

for split_name in splits:
    ds = load_dataset('sidleal/nurc_tts_24khz', split=split_name)
    metadata = ds.remove_columns(['audio'])
    for entry in metadata:
        entry['split'] = split_name
        all_metadata.append(entry)

df = pd.DataFrame(all_metadata)

print(f"Total metadata rows loaded: {len(df)}")
display(df.head())

In [4]:
unique_chars = set(''.join(df['text'].dropna().astype(str)))
sorted_chars = sorted(list(unique_chars))
print("Unique characters found:")
print(sorted_chars)

Unique characters found:
['\n', ' ', '%', ',', '-', '.', '/', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'á', 'â', 'ã', 'ç', 'é', 'ê', 'í', 'ó', 'ô', 'õ', 'ú']


#clean

In [ ]:
!pip install datasets huggingface_hub

In [6]:
from huggingface_hub import notebook_login
notebook_login()

In [7]:
ds = load_dataset('sidleal/nurc_tts_24khz')
print(ds)

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/78 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/111 [00:00<?, ?it/s]

DatasetDict({
    sao_paulo: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 120621
    })
    recife: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 200295
    })
})


In [8]:
import re
def clean_text(batch):
    cleaned_texts = []
    
    for text in batch["text"]:
        if text is None:
            cleaned_texts.append("")
            continue
            
        cleaned = re.sub(r'[\n%/]', '', text) 
        cleaned_texts.append(cleaned)
        
    return {"text": cleaned_texts}


In [9]:

#Apply the transformation using .map()
# batched=True processes the data in chunks, making it much faster
cleaned_dataset = ds.map(clean_text, batched=True)


Map:   0%|          | 0/120621 [00:00<?, ? examples/s]

Map:   0%|          | 0/200295 [00:00<?, ? examples/s]

In [10]:
print(cleaned_dataset)

DatasetDict({
    sao_paulo: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 120621
    })
    recife: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 200295
    })
})


In [11]:
metadata = cleaned_dataset['sao_paulo'].remove_columns(['audio'])
dfx = pd.DataFrame(metadata)

unique_chars = set(''.join(dfx['text'].dropna().astype(str)))
sorted_chars = sorted(list(unique_chars))
print("Unique characters found:")
print(sorted_chars)

Unique characters found:
[' ', ',', '-', '.', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'á', 'â', 'ã', 'ç', 'é', 'ê', 'í', 'ó', 'ô', 'õ', 'ú']


In [ ]:

# Upload it back to the Hugging Face Hub
cleaned_dataset.push_to_hub("sidleal/nurc_tts_24khz")